# Bias KoPolitic Transformer - Colab

3클래스 정치 편향도 분류용 Transformer 노트북입니다.

- KoPolitic 체크포인트를 우선 시도합니다.
- 공개 KoPolitic 체크포인트를 찾지 못하면 `klue/roberta-base`로 폴백합니다.
- 결과는 `metrics.json`과 `predictions.csv`로 저장합니다.


In [ ]:
!pip -q install transformers datasets accelerate sentencepiece evaluate scikit-learn pandas joblib


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json
import os
import time

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SPLIT_NAME = 'splits_label1_3class'
MODEL_DIR_NAME = 'classification/bias_transformer_kopolitic_3class/models'
OUTPUT_NAME = 'bias_kopolitic_transformer_3class'

PROJECT_ROOT = Path('/content/drive/MyDrive/sw_project/ai_features/political_bias_analysis')
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'original_data'
MODEL_DIR = PROJECT_ROOT / MODEL_DIR_NAME
MODEL_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR = DATA_DIR / SPLIT_NAME
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

RAW_TRAIN = RAW_DIR / 'train.csv'
RAW_TEST = RAW_DIR / 'test.csv'
if not RAW_TRAIN.exists():
    RAW_TRAIN = RAW_DIR / 'complete_train.csv'
if not RAW_TEST.exists():
    RAW_TEST = RAW_DIR / 'complete_test.csv'

TRAIN_CSV = SPLIT_DIR / 'train.csv'
VALID_CSV = SPLIT_DIR / 'valid.csv'
TEST_CSV = SPLIT_DIR / 'test.csv'

print('project_root =', PROJECT_ROOT)
print('data_dir =', DATA_DIR)
print('raw_dir =', RAW_DIR)
print('split_dir =', SPLIT_DIR)
print('model_dir =', MODEL_DIR)
print('raw_train =', RAW_TRAIN)
print('raw_test =', RAW_TEST)


def label1_to_three_class(value):
    value = int(value)
    if value in (1, 2):
        return 1
    if value == 3:
        return 2
    if value in (4, 5):
        return 3
    raise ValueError(f'Unexpected label1 value: {value}')


def compose_text(df):
    title = df['title'].fillna('').astype(str).str.strip()
    content = df['content'].fillna('').astype(str).str.strip()
    return (title + ' ' + content).str.strip()


def load_raw_frame(path):
    df = pd.read_csv(path)
    df = df.copy()
    df['text'] = compose_text(df)
    df['label'] = df['label1'].apply(label1_to_three_class).astype(int)
    cols = [c for c in ['seq', 'title', 'content', 'text', 'label'] if c in df.columns]
    return df[cols + [c for c in df.columns if c not in cols]]


In [ ]:
import subprocess
import time


def _log(msg):
    print(f'[split] {msg}')


def ensure_splits():
    split_start = time.perf_counter()
    raw_dir = DATA_DIR / 'original_data'
    train_source = raw_dir / 'complete_train.csv'
    test_source = raw_dir / 'complete_test.csv'

    _log(f'train_source = {train_source.exists()} | {train_source}')
    _log(f'test_source = {test_source.exists()} | {test_source}')

    if TRAIN_CSV.exists() and VALID_CSV.exists() and TEST_CSV.exists():
        _log('split files already exist; skipping creation')
        print(f'[split] elapsed = {time.perf_counter() - split_start:.2f}s')
        return

    if not train_source.exists() or not test_source.exists():
        raise FileNotFoundError('Raw CSV files are missing. Need data/original_data/complete_train.csv and complete_test.csv.')

    result = subprocess.run(
        [
            sys.executable,
            str(PROJECT_ROOT / 'prepare_label1_splits.py'),
            '--label-mode', 'three_class' if SPLIT_NAME.endswith('3class') else 'five_class',
            '--train-csv', str(train_source),
            '--test-csv', str(test_source),
            '--out-dir', str(SPLIT_DIR),
        ],
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    print(result.stderr)
    result.check_returncode()
    _log('split generation completed')
    print(f'[split] elapsed = {time.perf_counter() - split_start:.2f}s')


ensure_splits()


In [ ]:
def predict_bias(title, content):
    text = f'{title.strip()} {content.strip()}'.strip()
    encoded = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_LENGTH,
    )
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    trainer.model.to(device)
    encoded = {k: v.to(device) for k, v in encoded.items()}
    with torch.no_grad():
        logits = trainer.model(**encoded).logits.squeeze(0)
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
    order = np.argsort(probs)[::-1]
    top1 = int(order[0]) + 1
    top2 = int(order[1]) + 1
    return {
        'pred_label': top1,
        'top2_label': top2,
        'scores': {str(idx + 1): float(probs[idx]) for idx in range(len(probs))},
    }

# 예시
# predict_bias('기사 제목', '기사 본문')
